In [3]:
%pip install -q ultralytics

import json
import random
import shutil
import zipfile
import yaml
import torch
from pathlib import Path
from google.colab import drive
from ultralytics import YOLO

# 1. Drive Bağlantısı ve ZIP Açma (Colab'ın hızlı yerel SSD diskine)
drive.mount("/content/drive")
zip_path = Path("/content/drive/MyDrive/training_data/quadrant-enumeration-disease.zip")
if not zip_path.exists():
    raise FileNotFoundError(f"ZIP bulunamadı: {zip_path}")

extract_root = Path("/content/dentex_q3")
if extract_root.exists():
    shutil.rmtree(extract_root)
extract_root.mkdir(parents=True, exist_ok=True)

print("ZIP açılıyor, lütfen bekleyin...")
with zipfile.ZipFile(zip_path, "r") as zf:
    zf.extractall(extract_root)

# JSON dosyasını otomatik bul (iç içe klasör çıksa bile yakalar)
json_files = list(extract_root.rglob("train_quadrant_enumeration_disease.json"))
if not json_files:
    raise FileNotFoundError("ZIP içinde train_quadrant_enumeration_disease.json bulunamadı!")

coco_path = json_files[0]
BASE_DIR = coco_path.parent
images_src = BASE_DIR / "xrays"

image_files = sorted([f for f in images_src.glob("*.*") if f.suffix.lower() in [".png", ".jpg", ".jpeg"]])
print(f"Doğrulanan Toplam Röntgen Sayısı: {len(image_files)} (Beklenen: 705)")
assert len(image_files) == 705, f"Eksik dosya var! Bulunan: {len(image_files)}"

# 2. COCO JSON -> YOLO Etiketleri (Sadece 4 anomali sınıfı: category_id_3)
labels_dir = BASE_DIR / "labels"
labels_dir.mkdir(parents=True, exist_ok=True)

with open(coco_path, "r") as f:
    data = json.load(f)

images_info = {
    img["id"]: (Path(img["file_name"]).stem, img["width"], img["height"])
    for img in data["images"]
}

categories_3 = sorted(data["categories_3"], key=lambda x: x["id"])
cat_id_map = {cat["id"]: idx for idx, cat in enumerate(categories_3)}
class_names = {idx: cat["name"] for idx, cat in enumerate(categories_3)}
print("Sınıflar:", class_names)

yolo_labels = {stem: [] for stem, _, _ in images_info.values()}

for ann in data["annotations"]:
    img_id = ann["image_id"]
    if img_id not in images_info:
        continue
    stem, img_w, img_h = images_info[img_id]
    cls_idx = cat_id_map[ann["category_id_3"]]

    x_min, y_min, w, h = ann["bbox"]
    x_center = max(0.0, min(1.0, (x_min + w / 2.0) / img_w))
    y_center = max(0.0, min(1.0, (y_min + h / 2.0) / img_h))
    w_norm = max(0.0, min(1.0, w / img_w))
    h_norm = max(0.0, min(1.0, h / img_h))

    yolo_labels[stem].append(f"{cls_idx} {x_center:.6f} {y_center:.6f} {w_norm:.6f} {h_norm:.6f}")

for stem, lines in yolo_labels.items():
    with open(labels_dir / f"{stem}.txt", "w") as f:
        f.write("\n".join(lines))

# 3. Veriyi %70 Train / %15 Val / %15 Test Olarak Bölme
random.seed(42)
random.shuffle(image_files)

n_total = len(image_files)
n_train = int(n_total * 0.70)
n_val = int(n_total * 0.15)

splits = {
    "train": image_files[:n_train],
    "val": image_files[n_train:n_train + n_val],
    "test": image_files[n_train + n_val:]
}

for split_name, file_list in splits.items():
    img_dst = BASE_DIR / "images" / split_name
    lbl_dst = BASE_DIR / "labels" / split_name
    img_dst.mkdir(parents=True, exist_ok=True)
    lbl_dst.mkdir(parents=True, exist_ok=True)

    for img_path in file_list:
        shutil.copy(img_path, img_dst / img_path.name)
        lbl_path = labels_dir / f"{img_path.stem}.txt"
        if lbl_path.exists():
            shutil.copy(lbl_path, lbl_dst / lbl_path.name)

print(f"Bölünme Tamam -> Train: {len(splits['train'])} | Val: {len(splits['val'])} | Test: {len(splits['test'])}")

# 4. YAML Oluşturma
yaml_path = BASE_DIR / "disease_baseline.yaml"
dataset_cfg = {
    "path": str(BASE_DIR.resolve()),
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "names": class_names
}
with open(yaml_path, "w") as f:
    yaml.dump(dataset_cfg, f, default_flow_style=False, sort_keys=False)

# 5. Dümdüz (Brute-Force) YOLOv8n Eğitimi
model = YOLO("yolov8n.pt")
results = model.train(
    data=str(yaml_path),
    epochs=50,
    imgsz=640,
    batch=16,
    fliplr=0.0,
    device=0,
    project="/content/runs_q3",
    name="baseline_705_yolov8n"
)

# 6. Test Seti Değerlendirmesi (mAP50, mAP70 ve Sınıf Bazlı Rapor)
best_weights = Path(results.save_dir) / "weights" / "best.pt"
best_model = YOLO(str(best_weights))
metrics = best_model.val(data=str(yaml_path), split="test", project="/content/runs_q3", name="test_eval", plots=True)

all_ap = metrics.box.all_ap
map50 = metrics.box.map50
map70 = float(all_ap[:, 4].mean())  # IoU = 0.70
map50_95 = metrics.box.map

print("\n==========================================")
print("   GÖREV 3 GERÇEK BASELINE SONUÇLARI (705 GÖRÜNTÜ)")
print("==========================================")
print(f"Genel mAP50:     {map50:.4f}")
print(f"Genel mAP70:     {map70:.4f}")
print(f"Genel mAP50-95:  {map50_95:.4f}")
print(f"Genel Precision: {metrics.box.mp:.4f}")
print(f"Genel Recall:    {metrics.box.mr:.4f}")

print("\n--- SINIF BAZLI PERFORMANS ---")
for idx, name in class_names.items():
    p, r, ap50, ap50_95 = metrics.box.class_result(idx)
    ap70 = float(all_ap[idx, 4])
    print(f"{name:<20} | P: {p:.3f} | R: {r:.3f} | AP50: {ap50:.3f} | AP70: {ap70:.3f} | AP50-95: {ap50_95:.3f}")

# 7. Çıktıları ve Confusion Matrix'i Kalıcı Olarak Drive'a Yedekle
backup_dir = Path("/content/drive/MyDrive/dentex_q3_outputs/baseline")
backup_dir.mkdir(parents=True, exist_ok=True)
shutil.copy(best_weights, backup_dir / "best.pt")
for plot_name in ["confusion_matrix.png", "confusion_matrix_normalized.png", "PR_curve.png"]:
    p_file = Path(metrics.save_dir) / plot_name
    if p_file.exists():
        shutil.copy(p_file, backup_dir / plot_name)

print(f"\nModel ağırlığı ve grafikler Drive'a yedeklendi: {backup_dir}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
ZIP açılıyor, lütfen bekleyin...
Doğrulanan Toplam Röntgen Sayısı: 705 (Beklenen: 705)
Sınıflar: {0: 'Impacted', 1: 'Caries', 2: 'Periapical Lesion', 3: 'Deep Caries'}
Bölünme Tamam -> Train: 493 | Val: 105 | Test: 107
Ultralytics 8.4.218 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (NVIDIA L4, 22563MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dentex_q3/quadrant-enumeration-disease/disease_baseline.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, e